# Hands-on Decision Tree Analysis with Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mojiahui777/ba_course/blob/main/decision_tree_credit_risk_demo.ipynb)

> **Before you begin:** Select **File → Save a copy in Drive** to create your own copy.
>
> 1. Run the code cells in order, modify the parameters, and experiment with the results.
> 2. Your changes will be saved to your own Google Drive and will not affect the original notebook.
> 3. This 59-row dataset is for teaching only—not for real lending decisions.

## Learning objectives

By the end of this demonstration, you will be able to:

- frame a credit-risk question as a supervised classification problem;
- prepare numeric and coded categorical predictors for a decision tree;
- train and read an entropy-based decision tree;
- evaluate predictions with a baseline, a confusion matrix, and class-aware metrics; and
- explain why a small, imbalanced sample limits conclusions.

## 1. Define the business problem

**Goal:** Use applicant profiles to predict the recorded `CREDIT` class as a decision-support example for a bank loan manager.

A decision tree is a **supervised learning** method: it learns rules from rows that already have a known class. The legacy Weka slides describe the outcomes as Good and Bad credit risk, but this CSV does not document whether `0` or `1` means Good. Keep the numeric labels until a verified data dictionary supplies that mapping.

> A classroom model is not a fair, validated, or legally sufficient lending policy.

## 2. Import the Python libraries

In [ ]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

## 3. Load the data

Choose one option below:

- Keep `UPLOAD_FILE = False` to load the CSV from GitHub.
- Change it to `True` to display a file-upload button in Google Colab.

The local fallback lets the same notebook run when `Credit.csv` is beside it.

In [ ]:
# Option 1: Load the data directly from GitHub.
raw_url = "https://raw.githubusercontent.com/mojiahui777/ba_course/main/Credit.csv"
UPLOAD_FILE = False  # Change to True to upload a CSV in Colab

# Option 2: Display Colab's upload button when UPLOAD_FILE is True.
if UPLOAD_FILE:
    from google.colab import files

    uploaded = files.upload()
    df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
else:
    try:
        df = pd.read_csv(raw_url)
    except Exception:
        local_paths = [Path("Credit.csv")]
        local_path = next((path for path in local_paths if path.exists()), None)
        if local_path is None:
            raise FileNotFoundError(
                "Could not download Credit.csv. Set UPLOAD_FILE = True in Colab "
                "or place the CSV beside the notebook."
            )
        df = pd.read_csv(local_path)

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 4. Understand the variables

| Column | Role | Meaning |
|---|---|---|
| `OBS#` | ID | Observation identifier; exclude from the model |
| `CHK_ACCT` | Categorical input | Checking-account category code |
| `DURATION` | Numeric input | Loan duration in months |
| `AGE` | Numeric input | Applicant age in years |
| `Amount` | Numeric input | Requested credit amount |
| `JOB` | Categorical input | Job category code |
| `NUM_DEPENDENTS` | Categorical input | Number-of-dependents code |
| `TELEPHONE` | Categorical input | Telephone category code |
| `FOREIGN` | Categorical input | Foreign-worker category code |
| `CREDIT` | Target | Recorded credit class (`0` or `1`) |

The legacy slides use 1-based descriptions for some categories while this file contains 0-based codes. We can model the codes as categories, but should not attach names to them without a matching data dictionary.

In [ ]:
expected_columns = [
    "OBS#", "CHK_ACCT", "DURATION", "AGE", "Amount",
    "JOB", "NUM_DEPENDENTS", "TELEPHONE", "FOREIGN", "CREDIT",
]
assert list(df.columns) == expected_columns, "The CSV columns do not match the teaching dataset."

display(df.describe(include="all").round(2))

## 5. Check data quality and the target

Before fitting a model, check missing values and class balance. A **majority-class baseline** predicts the most frequent class for every row; a useful model should be compared with that simple rule.

In [ ]:
target_column = "CREDIT"
id_column = "OBS#"

print("Missing values per column:")
display(df.isna().sum())

target_counts = df[target_column].value_counts().sort_index()
display(target_counts.rename("count").to_frame())
print(f"Majority-class baseline accuracy: {target_counts.max() / target_counts.sum():.3f}")

target_counts.plot(kind="bar", color=["#CC0000", "#5B6573"], rot=0)
plt.xlabel("CREDIT class")
plt.ylabel("Number of observations")
plt.title("Target Class Distribution")
plt.tight_layout()
plt.show()

## 6. Select predictors and prepare the data

Remove the ID and target from the predictors. One-hot encode category codes so the tree does not treat them as measurements with equal numerical spacing. Numeric fields pass through unchanged; decision trees do not require feature scaling.

In [ ]:
categorical_columns = [
    "CHK_ACCT", "JOB", "NUM_DEPENDENTS", "TELEPHONE", "FOREIGN"
]
feature_columns = [
    column for column in df.columns if column not in [id_column, target_column]
]
numeric_columns = [
    column for column in feature_columns if column not in categorical_columns
]

X = df[feature_columns].copy()
y = df[target_column].copy()

preprocessor = ColumnTransformer(
    transformers=[
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("number", "passthrough", numeric_columns),
    ],
    verbose_feature_names_out=False,
)

print("Categorical predictors:", categorical_columns)
print("Numeric predictors:", numeric_columns)

## 7. Create an 80/20 train-test split

Following the legacy Weka case, use 80% of the rows for training and 20% for testing. `stratify=y` keeps the class proportions similar, and `random_state=42` makes the split reproducible.

With only 59 rows, the test set has just 12 observations. A different split can noticeably change the results.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows:  {len(X_test)}")
print("Training class counts:")
display(y_train.value_counts().sort_index().rename("count").to_frame())
print("Testing class counts:")
display(y_test.value_counts().sort_index().rename("count").to_frame())

## 8. Train a decision tree

A tree repeatedly asks feature-based questions that make the resulting groups more class-pure. We use:

- `criterion="entropy"` as a teaching analogue to Weka's information-gain-based J48;
- `max_depth=3` so the displayed tree remains readable; and
- `random_state=42` for reproducibility.

Scikit-learn's tree is **not** an exact implementation of Weka J48/C4.5.

In [ ]:
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    (
        "tree",
        DecisionTreeClassifier(
            criterion="entropy",
            max_depth=3,
            random_state=42,
        ),
    ),
])

model.fit(X_train, y_train)
print("Model fitted.")

## 9. Evaluate holdout predictions

Compare the tree with the majority baseline, then inspect balanced accuracy and the per-class report.

- **Accuracy** is the share of all predictions that are correct.
- **Balanced accuracy** gives equal weight to recall for each class.
- **Precision** asks how often a predicted class is correct.
- **Recall** asks how often the model finds members of an actual class.

In [ ]:
y_pred = model.predict(X_test)
baseline_accuracy = y_train.value_counts(normalize=True).max()

print(f"Test accuracy:              {accuracy_score(y_test, y_pred):.3f}")
print(f"Test balanced accuracy:     {balanced_accuracy_score(y_test, y_pred):.3f}")
print(f"Training majority baseline: {baseline_accuracy:.3f}")
print("\nPer-class report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 10. Read the confusion matrix

Rows are actual classes and columns are predicted classes. The off-diagonal cells are errors. Keep the numeric class labels until their Good/Bad meaning is verified.

In [ ]:
class_labels = sorted(y.unique())
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=class_labels,
    display_labels=[str(value) for value in class_labels],
    cmap="Reds",
    colorbar=False,
)
plt.title("Decision Tree: Holdout Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.tight_layout()
plt.show()

## 11. Visualize and read the tree

Start at the top node. If a condition is true, move left; otherwise, move right. At each node:

- `entropy` measures class mixture (lower is purer);
- `samples` is the number of training rows at that node;
- `value` gives the class counts; and
- `class` is the node's predicted class.

In [ ]:
tree_model = model.named_steps["tree"]
feature_names = model.named_steps["preprocess"].get_feature_names_out()

plt.figure(figsize=(18, 9))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=[str(value) for value in tree_model.classes_],
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("Entropy-Based Decision Tree (Maximum Depth 3)")
plt.tight_layout()
plt.show()

## 12. Translate the tree into text rules

In [ ]:
print(export_text(tree_model, feature_names=list(feature_names)))

## 13. Explore feature importance

Feature importance summarizes how much each transformed feature reduced impurity across this fitted tree. It is model-specific—not proof of causality, fairness, or a stable ranking for a larger population.

In [ ]:
importance = pd.Series(
    tree_model.feature_importances_, index=feature_names
).sort_values(ascending=False)
used_features = importance[importance > 0].sort_values()

display(used_features.sort_values(ascending=False).round(3).rename("importance").to_frame())
used_features.plot(kind="barh", figsize=(8, 4), color="#CC0000")
plt.xlabel("Impurity-based importance")
plt.ylabel("Transformed feature")
plt.title("Features Used by the Fitted Tree")
plt.tight_layout()
plt.show()

## 14. Experiment: can class weighting change the errors?

The target is imbalanced. `class_weight="balanced"` gives errors on the less frequent class more influence during training. Compare the resulting confusion matrix with the original one; do not assume the weighted model is automatically better.

In [ ]:
weighted_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    (
        "tree",
        DecisionTreeClassifier(
            criterion="entropy",
            max_depth=3,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])
weighted_model.fit(X_train, y_train)
weighted_pred = weighted_model.predict(X_test)

comparison = pd.DataFrame(
    {
        "Model": ["Original tree", "Class-weighted tree"],
        "Accuracy": [
            accuracy_score(y_test, y_pred),
            accuracy_score(y_test, weighted_pred),
        ],
        "Balanced accuracy": [
            balanced_accuracy_score(y_test, y_pred),
            balanced_accuracy_score(y_test, weighted_pred),
        ],
    }
).set_index("Model")
display(comparison.round(3))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    weighted_pred,
    labels=class_labels,
    display_labels=[str(value) for value in class_labels],
    cmap="Reds",
    colorbar=False,
)
plt.title("Class-Weighted Tree: Holdout Confusion Matrix")
plt.tight_layout()
plt.show()

## 15. Discuss the result

- Does the original tree outperform the majority-class baseline?
- Which actual class is hardest for the original tree to identify?
- How does class weighting change the types of errors?
- What would a false positive and false negative mean after the class mapping is verified?
- Why might another 80/20 split produce a different tree and confusion matrix?
- Which category codes require a matching data dictionary before interpretation?
- Why are 59 historical rows insufficient for a real approval policy?

**Takeaway:** A decision tree makes rule-based predictions that are easy to inspect, but interpretability does not guarantee accuracy, fairness, stability, or suitability for a high-impact decision.